<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **Complejidad computacional, benchmarking y profiling**

## De “funciona” a “sé cuánto cuesta y dónde optimizar”

En este laboratorio dejamos de evaluar el código únicamente por su resultado.

Un pipeline profesional también debe analizarse en términos de:

- tiempo;
- memoria;
- escalabilidad;
- complejidad;
- cuellos de botella;
- variabilidad de las mediciones;
- coste de las decisiones de implementación.

La secuencia del laboratorio será:

> **implementar → observar lentitud → medir → localizar cuello de botella → formular hipótesis → optimizar → volver a medir → validar que el resultado sigue siendo correcto**

No optimizaremos “por intuición”.

## Objetivos

Al finalizar deberías poder:

1. Explicar intuitivamente la complejidad temporal.
2. Reconocer patrones O(1), O(n), O(n log n) y O(n²).
3. Diferenciar complejidad teórica y tiempo real.
4. Construir benchmarks reproducibles.
5. Entender por qué una única medición no es suficiente.
6. Utilizar `time.perf_counter`.
7. Utilizar `timeit`.
8. Analizar media, mediana, dispersión y speedup.
9. Estudiar escalabilidad experimentalmente.
10. Medir memoria.
11. Detectar copias innecesarias.
12. Utilizar `cProfile`.
13. Interpretar tiempo acumulado y tiempo propio.
14. Utilizar profiling por línea cuando esté disponible.
15. Comparar loops y vectorización.
16. Detectar UDFs costosas.
17. Comparar Pandas y Polars correctamente.
18. Separar coste de conversión y coste de cálculo.
19. Construir una metodología profesional de optimización.

## 1. Preparación

In [ ]:
import time
import timeit
import cProfile
import pstats
import io
import tracemalloc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

try:
    import polars as pl
    POLARS_DISPONIBLE = True
    print("Polars:", pl.__version__)
except ImportError:
    POLARS_DISPONIBLE = False
    print("Polars no instalado.")

print("Pandas:", pd.__version__)

# 2. Complejidad computacional: intuición

La notación Big-O describe cómo crece aproximadamente el coste cuando aumenta el tamaño de entrada.

Ejemplos habituales:

| Complejidad | Interpretación |
|---|---|
| O(1) | coste aproximadamente constante |
| O(log n) | crecimiento muy lento |
| O(n) | crecimiento proporcional |
| O(n log n) | típico de algoritmos eficientes de ordenación |
| O(n²) | crecimiento cuadrático |
| O(2ⁿ) | crecimiento explosivo |

Big-O **no representa segundos**. Describe comportamiento asintótico.

# 3. O(1): acceso directo

In [ ]:
datos = np.arange(1_000_000)

inicio = time.perf_counter()
valor = datos[500_000]
fin = time.perf_counter()

print("Valor:", valor)
print("Tiempo:", (fin - inicio) * 1e6, "µs")

Aunque aumente el array, acceder a una posición concreta no requiere recorrer todos los elementos.

# 4. O(n): recorrido lineal

In [ ]:
def suma_manual(vector):
    total = 0
    for valor in vector:
        total += valor
    return total

for n in [10_000, 50_000, 100_000, 500_000]:
    vector = np.arange(n)
    inicio = time.perf_counter()
    suma_manual(vector)
    tiempo = time.perf_counter() - inicio
    print(n, round(tiempo * 1000, 3), "ms")

# 5. Visualizar escalabilidad lineal

In [ ]:
tamanos = np.array([
    10_000,
    25_000,
    50_000,
    100_000,
    250_000,
    500_000
])

tiempos_lineales = []

for n in tamanos:
    vector = np.arange(n)

    inicio = time.perf_counter()
    suma_manual(vector)
    tiempos_lineales.append(
        time.perf_counter() - inicio
    )

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(
    tamanos,
    np.array(tiempos_lineales) * 1000,
    marker="o"
)
ax.set(
    title="Escalabilidad experimental de un recorrido",
    xlabel="n",
    ylabel="Tiempo (ms)"
)
plt.show()

### Interpretación

Esperamos una tendencia aproximadamente lineal, aunque las mediciones reales contienen ruido por hardware, caché, sistema operativo y otras tareas.

# 6. O(n²): comparación por pares

In [ ]:
def contar_pares(vector):
    contador = 0
    for i in vector:
        for j in vector:
            if i < j:
                contador += 1
    return contador

tamanos_quad = np.array([
    100, 200, 400, 800, 1200
])

tiempos_quad = []

for n in tamanos_quad:
    vector = np.arange(n)

    inicio = time.perf_counter()
    contar_pares(vector)
    tiempos_quad.append(
        time.perf_counter() - inicio
    )

    print(
        n,
        round(tiempos_quad[-1] * 1000, 3),
        "ms"
    )

# 7. Visualizar crecimiento cuadrático

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(
    tamanos_quad,
    np.array(tiempos_quad) * 1000,
    marker="o"
)

ax.set(
    title="Crecimiento de un algoritmo O(n²)",
    xlabel="n",
    ylabel="Tiempo (ms)"
)

plt.show()

### Interpretación

Duplicar `n` en un algoritmo cuadrático puede multiplicar aproximadamente por cuatro el número de operaciones dominantes.

# 8. Error frecuente: medir una sola vez

In [ ]:
vector = np.arange(500_000)

inicio = time.perf_counter()
_ = suma_manual(vector)
tiempo_unico = time.perf_counter() - inicio

print(
    "Una única medición:",
    tiempo_unico * 1000,
    "ms"
)

### Problema

Una sola medición puede verse afectada por ruido.

Necesitamos repeticiones.

# 9. Función reusable de benchmark

In [ ]:
def benchmark(
    funcion,
    repeticiones=10,
    calentamiento=2
):
    for _ in range(calentamiento):
        funcion()

    tiempos = []

    for _ in range(repeticiones):
        inicio = time.perf_counter()
        funcion()
        tiempos.append(
            time.perf_counter() - inicio
        )

    tiempos = np.array(tiempos)

    return {
        "media_ms": tiempos.mean() * 1000,
        "mediana_ms": np.median(tiempos) * 1000,
        "std_ms": tiempos.std() * 1000,
        "min_ms": tiempos.min() * 1000,
        "max_ms": tiempos.max() * 1000,
        "raw": tiempos
    }

# 10. Aplicar benchmark

In [ ]:
resultado_bench = benchmark(
    lambda: suma_manual(vector),
    repeticiones=20
)

{
    k: v
    for k, v in resultado_bench.items()
    if k != "raw"
}

# 11. Visualizar distribución de tiempos

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(
    resultado_bench["raw"] * 1000,
    bins=10
)

ax.axvline(
    resultado_bench["media_ms"],
    linestyle="--",
    label="Media"
)

ax.axvline(
    resultado_bench["mediana_ms"],
    linestyle=":",
    label="Mediana"
)

ax.set(
    title="Distribución de tiempos del benchmark",
    xlabel="Tiempo (ms)",
    ylabel="Frecuencia"
)

ax.legend()
plt.show()

### Interpretación

La dispersión también es información.

Dos implementaciones con la misma media pueden tener distinta estabilidad.

# 12. `timeit`

In [ ]:
tiempos_timeit = timeit.repeat(
    stmt="suma_manual(vector)",
    globals={
        "suma_manual": suma_manual,
        "vector": vector
    },
    repeat=7,
    number=3
)

print(
    np.array(tiempos_timeit) * 1000
)

`timeit` es útil para microbenchmarks porque automatiza repeticiones y reduce parte del ruido de medición.

# 13. Loop Python vs NumPy

In [ ]:
vector_float = rng.normal(
    size=1_000_000
)

def cuadrado_loop():
    salida = []
    for x in vector_float:
        salida.append(x ** 2)
    return salida

def cuadrado_numpy():
    return vector_float ** 2

bench_loop = benchmark(
    cuadrado_loop,
    repeticiones=7
)

bench_numpy = benchmark(
    cuadrado_numpy,
    repeticiones=7
)

comparacion_vectorizacion = pd.DataFrame({
    "metodo": [
        "Loop Python",
        "NumPy vectorizado"
    ],
    "media_ms": [
        bench_loop["media_ms"],
        bench_numpy["media_ms"]
    ]
})

comparacion_vectorizacion

# 14. Calcular speedup

In [ ]:
speedup = (
    bench_loop["media_ms"]
    / bench_numpy["media_ms"]
)

print(
    f"Speedup aproximado: {speedup:.2f}x"
)

# 15. Visualizar comparación

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    comparacion_vectorizacion["metodo"],
    comparacion_vectorizacion["media_ms"]
)

ax.set(
    title="Loop Python vs vectorización",
    ylabel="Tiempo medio (ms)"
)

plt.show()

### Interpretación

La vectorización desplaza el trabajo desde el intérprete Python hacia implementaciones optimizadas de bajo nivel.

Pero siempre debemos validar que el resultado sea equivalente.

# 16. Validar resultados

In [ ]:
resultado_loop = np.array(
    cuadrado_loop()
)

resultado_numpy = cuadrado_numpy()

print(
    "Resultados equivalentes:",
    np.allclose(
        resultado_loop,
        resultado_numpy
    )
)

# 17. Dataset tabular para profiling

In [ ]:
N = 500_000

df = pd.DataFrame({
    "cliente_id": rng.integers(
        1,
        50_001,
        N
    ),
    "region": rng.choice(
        ["Norte", "Sur", "Este", "Oeste"],
        N
    ),
    "unidades": rng.integers(
        1,
        10,
        N
    ),
    "precio": rng.uniform(
        5,
        300,
        N
    ),
    "descuento": rng.choice(
        [0.0, 0.05, 0.10, 0.20],
        N
    )
})

df.head()

# 18. Implementación deliberadamente ineficiente

In [ ]:
def calcular_importe_apply(datos):
    copia = datos.copy()

    copia["importe"] = copia.apply(
        lambda fila:
            fila["unidades"]
            * fila["precio"]
            * (1 - fila["descuento"]),
        axis=1
    )

    return copia

# 19. Medir `apply(axis=1)`

In [ ]:
bench_apply = benchmark(
    lambda: calcular_importe_apply(
        df.head(100_000)
    ),
    repeticiones=5,
    calentamiento=1
)

{
    k: v
    for k, v in bench_apply.items()
    if k != "raw"
}

### Diagnóstico inicial

`apply(axis=1)` ejecuta lógica Python fila a fila.

Antes de aceptar su coste, debemos comprobar si la operación puede expresarse vectorialmente.

# 20. Versión vectorizada

In [ ]:
def calcular_importe_vectorizado(datos):
    copia = datos.copy()

    copia["importe"] = (
        copia["unidades"]
        * copia["precio"]
        * (
            1
            - copia["descuento"]
        )
    )

    return copia

# 21. Benchmark `apply` vs vectorización

In [ ]:
muestra_df = df.head(100_000)

bench_apply = benchmark(
    lambda:
        calcular_importe_apply(muestra_df),
    repeticiones=5,
    calentamiento=1
)

bench_vec = benchmark(
    lambda:
        calcular_importe_vectorizado(muestra_df),
    repeticiones=10
)

comparacion_apply = pd.DataFrame({
    "metodo": [
        "apply(axis=1)",
        "Vectorizado"
    ],
    "media_ms": [
        bench_apply["media_ms"],
        bench_vec["media_ms"]
    ]
})

comparacion_apply

# 22. Visualizar

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    comparacion_apply["metodo"],
    comparacion_apply["media_ms"]
)

ax.set(
    title="Pandas: apply fila a fila vs vectorización",
    ylabel="Tiempo medio (ms)"
)

plt.show()

# 23. Validar equivalencia

In [ ]:
a = calcular_importe_apply(
    muestra_df
)["importe"].to_numpy()

b = calcular_importe_vectorizado(
    muestra_df
)["importe"].to_numpy()

print(
    "Equivalentes:",
    np.allclose(a, b)
)

# 24. Escalabilidad de ambas implementaciones

In [ ]:
tamanos_df = [
    5_000,
    10_000,
    25_000,
    50_000,
    100_000
]

resultados_escala = []

for n in tamanos_df:
    muestra = df.head(n)

    t_apply = benchmark(
        lambda m=muestra:
            calcular_importe_apply(m),
        repeticiones=3,
        calentamiento=0
    )["media_ms"]

    t_vec = benchmark(
        lambda m=muestra:
            calcular_importe_vectorizado(m),
        repeticiones=5,
        calentamiento=1
    )["media_ms"]

    resultados_escala.append({
        "n": n,
        "apply_ms": t_apply,
        "vectorizado_ms": t_vec
    })

escala_df = pd.DataFrame(
    resultados_escala
)

escala_df

# 25. Visualizar escalabilidad

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(
    escala_df["n"],
    escala_df["apply_ms"],
    marker="o",
    label="apply"
)

ax.plot(
    escala_df["n"],
    escala_df["vectorizado_ms"],
    marker="o",
    label="vectorizado"
)

ax.set(
    title="Escalabilidad experimental",
    xlabel="Número de filas",
    ylabel="Tiempo (ms)"
)

ax.legend()
plt.show()

### Interpretación

No basta con saber qué implementación gana en 5.000 filas.

Queremos observar cómo se comporta la diferencia cuando `n` aumenta.

# 26. Profiling con `cProfile`

In [ ]:
def pipeline_lento():
    datos = calcular_importe_apply(
        df.head(50_000)
    )

    return (
        datos.groupby(
            "region",
            as_index=False
        )
        .agg(
            facturacion=("importe", "sum"),
            ticket=("importe", "mean")
        )
    )

profiler = cProfile.Profile()
profiler.enable()

resultado = pipeline_lento()

profiler.disable()

stream = io.StringIO()

stats = pstats.Stats(
    profiler,
    stream=stream
)

stats.sort_stats(
    "cumulative"
)

stats.print_stats(20)

print(
    stream.getvalue()
)

## Cómo leer `cProfile`

Columnas habituales:

- `ncalls`: número de llamadas;
- `tottime`: tiempo ejecutado dentro de la función;
- `percall`: coste medio;
- `cumtime`: tiempo acumulado incluyendo funciones llamadas.

Para encontrar un cuello de botella suele ser especialmente útil ordenar por `cumtime`.

# 27. Optimizar después del profiling

In [ ]:
def pipeline_optimizado():
    datos = calcular_importe_vectorizado(
        df.head(50_000)
    )

    return (
        datos.groupby(
            "region",
            as_index=False
        )
        .agg(
            facturacion=("importe", "sum"),
            ticket=("importe", "mean")
        )
    )

bench_lento = benchmark(
    pipeline_lento,
    repeticiones=3,
    calentamiento=0
)

bench_optimizado = benchmark(
    pipeline_optimizado,
    repeticiones=7
)

print(
    "Lento:",
    bench_lento["media_ms"],
    "ms"
)

print(
    "Optimizado:",
    bench_optimizado["media_ms"],
    "ms"
)

print(
    "Speedup:",
    bench_lento["media_ms"]
    / bench_optimizado["media_ms"]
)

### Metodología

No hemos optimizado aleatoriamente.

1. medimos;
2. perfilamos;
3. localizamos una operación costosa;
4. sustituimos esa operación;
5. volvemos a medir.

# 28. Medir memoria con `memory_usage`

In [ ]:
memoria_columnas = (
    df.memory_usage(
        deep=True
    )
    .sort_values(
        ascending=False
    )
)

memoria_columnas

# 29. Visualizar memoria por columna

In [ ]:
memoria_mb = (
    memoria_columnas
    / 1024**2
)

fig, ax = plt.subplots(figsize=(8, 4))

ax.barh(
    memoria_mb.index.astype(str),
    memoria_mb.values
)

ax.set(
    title="Memoria por columna",
    xlabel="MB",
    ylabel="Columna"
)

plt.show()

### Interpretación

El profiling de memoria puede revelar que columnas categóricas almacenadas como `object` consumen una parte importante del DataFrame.

# 30. Optimizar dtype

In [ ]:
df_opt = df.copy()

df_opt["region"] = (
    df_opt["region"]
    .astype("category")
)

mem_antes = (
    df.memory_usage(
        deep=True
    ).sum()
    / 1024**2
)

mem_despues = (
    df_opt.memory_usage(
        deep=True
    ).sum()
    / 1024**2
)

print(
    "Antes:",
    round(mem_antes, 2),
    "MB"
)

print(
    "Después:",
    round(mem_despues, 2),
    "MB"
)

print(
    "Reducción:",
    round(
        100
        * (
            1
            - mem_despues / mem_antes
        ),
        2
    ),
    "%"
)

# 31. Medir pico de memoria con `tracemalloc`

In [ ]:
def medir_memoria(funcion):
    tracemalloc.start()

    inicio = time.perf_counter()
    resultado = funcion()
    tiempo = time.perf_counter() - inicio

    actual, pico = (
        tracemalloc.get_traced_memory()
    )

    tracemalloc.stop()

    return {
        "tiempo_ms": tiempo * 1000,
        "actual_mb": actual / 1024**2,
        "pico_mb": pico / 1024**2,
        "resultado": resultado
    }


mem_vec = medir_memoria(
    lambda:
        calcular_importe_vectorizado(
            df.head(200_000)
        )
)

{
    k: v
    for k, v in mem_vec.items()
    if k != "resultado"
}

`tracemalloc` permite observar asignaciones de memoria Python durante la operación.

La memoria reportada no equivale necesariamente a toda la memoria física utilizada por librerías nativas, pero es útil para comparar patrones Python.

# 32. Copias innecesarias

In [ ]:
def con_copias(datos):
    a = datos.copy()
    b = a.copy()
    c = b.copy()

    c["importe"] = (
        c["unidades"]
        * c["precio"]
    )

    return c


def copia_unica(datos):
    resultado = datos.copy()

    resultado["importe"] = (
        resultado["unidades"]
        * resultado["precio"]
    )

    return resultado


mem_copias = medir_memoria(
    lambda:
        con_copias(
            df.head(200_000)
        )
)

mem_unica = medir_memoria(
    lambda:
        copia_unica(
            df.head(200_000)
        )
)

pd.DataFrame({
    "metodo": [
        "Varias copias",
        "Una copia"
    ],
    "pico_mb": [
        mem_copias["pico_mb"],
        mem_unica["pico_mb"]
    ],
    "tiempo_ms": [
        mem_copias["tiempo_ms"],
        mem_unica["tiempo_ms"]
    ]
})

# 33. Visualizar tiempo y memoria

In [ ]:
comparacion_copias = pd.DataFrame({
    "metodo": [
        "Varias copias",
        "Una copia"
    ],
    "pico_mb": [
        mem_copias["pico_mb"],
        mem_unica["pico_mb"]
    ],
    "tiempo_ms": [
        mem_copias["tiempo_ms"],
        mem_unica["tiempo_ms"]
    ]
})

fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    comparacion_copias["metodo"],
    comparacion_copias["pico_mb"]
)

ax.set(
    title="Pico de memoria según estrategia de copia",
    ylabel="MB"
)

plt.show()

# 34. Complejidad de búsqueda: lista vs set

In [ ]:
valores = list(
    range(1_000_000)
)

valores_set = set(
    valores
)

objetivo = 999_999

bench_lista = benchmark(
    lambda:
        objetivo in valores,
    repeticiones=20
)

bench_set = benchmark(
    lambda:
        objetivo in valores_set,
    repeticiones=20
)

pd.DataFrame({
    "estructura": [
        "list",
        "set"
    ],
    "media_ms": [
        bench_lista["media_ms"],
        bench_set["media_ms"]
    ]
})

### Interpretación

La elección de estructura de datos también es una decisión algorítmica.

Buscar pertenencia en una lista requiere normalmente un recorrido; un `set` utiliza hashing y ofrece coste medio aproximadamente constante.

# 35. Optimización que NO merece la pena

No toda operación lenta necesita optimización.

Preguntas previas:

1. ¿aparece realmente en el perfil?
2. ¿se ejecuta muchas veces?
3. ¿el dataset crecerá?
4. ¿la mejora compensa la pérdida de legibilidad?
5. ¿existe un SLA o requisito de memoria?
6. ¿la optimización cambia mantenibilidad?

Optimizar código irrelevante puede ser una pérdida de tiempo.

# 36. Preparar Pandas vs Polars

In [ ]:
if POLARS_DISPONIBLE:
    df_pl = pl.from_pandas(
        df
    )

    print(
        df_pl.head()
    )
else:
    print(
        "Instalar Polars para continuar."
    )

# 37. Benchmark cálculo de columna

In [ ]:
def pandas_calculo():
    return (
        df["unidades"]
        * df["precio"]
        * (
            1
            - df["descuento"]
        )
    )


def polars_calculo():
    if not POLARS_DISPONIBLE:
        return None

    return (
        df_pl
        .select(
            (
                pl.col("unidades")
                * pl.col("precio")
                * (
                    1
                    - pl.col("descuento")
                )
            )
            .alias("importe")
        )
    )


bench_pd_calc = benchmark(
    pandas_calculo,
    repeticiones=10
)

if POLARS_DISPONIBLE:
    bench_pl_calc = benchmark(
        polars_calculo,
        repeticiones=10
    )
else:
    bench_pl_calc = {
        "media_ms": np.nan
    }

comparacion_motores = pd.DataFrame({
    "motor": [
        "Pandas",
        "Polars"
    ],
    "media_ms": [
        bench_pd_calc["media_ms"],
        bench_pl_calc["media_ms"]
    ]
})

comparacion_motores

# 38. Visualizar

In [ ]:
if POLARS_DISPONIBLE:
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.bar(
        comparacion_motores["motor"],
        comparacion_motores["media_ms"]
    )

    ax.set(
        title="Cálculo vectorizado — Pandas vs Polars",
        ylabel="Tiempo medio (ms)"
    )

    plt.show()

# 39. Validar equivalencia

In [ ]:
if POLARS_DISPONIBLE:
    resultado_pd = (
        pandas_calculo()
        .to_numpy()
    )

    resultado_pl = (
        polars_calculo()["importe"]
        .to_numpy()
    )

    print(
        "Equivalentes:",
        np.allclose(
            resultado_pd,
            resultado_pl
        )
    )

# 40. Benchmark GroupBy

In [ ]:
def pandas_groupby():
    datos = df.assign(
        importe=(
            df["unidades"]
            * df["precio"]
            * (
                1
                - df["descuento"]
            )
        )
    )

    return (
        datos.groupby(
            "region",
            as_index=False
        )
        .agg(
            facturacion=("importe", "sum"),
            operaciones=("cliente_id", "size"),
            clientes=("cliente_id", "nunique")
        )
    )


def polars_groupby():
    if not POLARS_DISPONIBLE:
        return None

    return (
        df_pl
        .with_columns(
            (
                pl.col("unidades")
                * pl.col("precio")
                * (
                    1
                    - pl.col("descuento")
                )
            )
            .alias("importe")
        )
        .group_by("region")
        .agg(
            pl.col("importe")
            .sum()
            .alias("facturacion"),

            pl.len()
            .alias("operaciones"),

            pl.col("cliente_id")
            .n_unique()
            .alias("clientes")
        )
    )


bench_pd_group = benchmark(
    pandas_groupby,
    repeticiones=7
)

if POLARS_DISPONIBLE:
    bench_pl_group = benchmark(
        polars_groupby,
        repeticiones=7
    )
else:
    bench_pl_group = {
        "media_ms": np.nan
    }

pd.DataFrame({
    "motor": [
        "Pandas",
        "Polars"
    ],
    "media_ms": [
        bench_pd_group["media_ms"],
        bench_pl_group["media_ms"]
    ]
})

# 41. Coste de conversión

In [ ]:
if POLARS_DISPONIBLE:
    bench_conversion = benchmark(
        lambda:
            pl.from_pandas(df),
        repeticiones=7
    )

    print(
        "Conversión Pandas → Polars:",
        round(
            bench_conversion["media_ms"],
            3
        ),
        "ms"
    )

### Interpretación

No es válido comparar:

`conversión + cálculo Polars`

contra:

`solo cálculo Pandas`

sin declarar qué estamos midiendo.

El benchmark debe reflejar el pipeline real.

# 42. Eager vs lazy en Polars

In [ ]:
if POLARS_DISPONIBLE:
    def pipeline_eager():
        return (
            df_pl
            .filter(
                pl.col("precio") > 100
            )
            .with_columns(
                (
                    pl.col("unidades")
                    * pl.col("precio")
                )
                .alias("importe")
            )
            .group_by("region")
            .agg(
                pl.col("importe")
                .sum()
                .alias("facturacion")
            )
        )

    consulta_lazy = (
        df_pl
        .lazy()
        .filter(
            pl.col("precio") > 100
        )
        .with_columns(
            (
                pl.col("unidades")
                * pl.col("precio")
            )
            .alias("importe")
        )
        .group_by("region")
        .agg(
            pl.col("importe")
            .sum()
            .alias("facturacion")
        )
    )

    def pipeline_lazy():
        return consulta_lazy.collect()

    bench_eager = benchmark(
        pipeline_eager,
        repeticiones=7
    )

    bench_lazy = benchmark(
        pipeline_lazy,
        repeticiones=7
    )

    print(
        "Eager:",
        bench_eager["media_ms"],
        "ms"
    )

    print(
        "Lazy:",
        bench_lazy["media_ms"],
        "ms"
    )

# 43. Inspeccionar plan lazy

In [ ]:
if POLARS_DISPONIBLE:
    print(
        consulta_lazy.explain(
            optimized=True
        )
    )

Lazy execution no significa automáticamente que cualquier consulta será mucho más rápida.

Su principal ventaja es que el motor puede observar y optimizar el plan completo.

# 44. Escalabilidad Pandas vs Polars

In [ ]:
if POLARS_DISPONIBLE:
    tamanos_motor = [
        10_000,
        50_000,
        100_000,
        250_000,
        500_000
    ]

    filas = []

    for n in tamanos_motor:
        pd_n = df.head(n)
        pl_n = df_pl.head(n)

        def f_pd():
            return (
                pd_n.groupby(
                    "region",
                    as_index=False
                )
                .agg(
                    precio_medio=(
                        "precio",
                        "mean"
                    )
                )
            )

        def f_pl():
            return (
                pl_n
                .group_by("region")
                .agg(
                    pl.col("precio")
                    .mean()
                    .alias(
                        "precio_medio"
                    )
                )
            )

        t_pd = benchmark(
            f_pd,
            repeticiones=5,
            calentamiento=1
        )["media_ms"]

        t_pl = benchmark(
            f_pl,
            repeticiones=5,
            calentamiento=1
        )["media_ms"]

        filas.append({
            "n": n,
            "Pandas": t_pd,
            "Polars": t_pl
        })

    escala_motores = pd.DataFrame(
        filas
    )

    escala_motores

# 45. Visualizar escalabilidad de motores

In [ ]:
if POLARS_DISPONIBLE:
    fig, ax = plt.subplots(figsize=(8, 4))

    ax.plot(
        escala_motores["n"],
        escala_motores["Pandas"],
        marker="o",
        label="Pandas"
    )

    ax.plot(
        escala_motores["n"],
        escala_motores["Polars"],
        marker="o",
        label="Polars"
    )

    ax.set(
        title="Escalabilidad experimental",
        xlabel="Número de filas",
        ylabel="Tiempo medio (ms)"
    )

    ax.legend()
    plt.show()

### Interpretación

Este gráfico es más informativo que un benchmark sobre un único tamaño: permite estudiar cómo evoluciona el coste al crecer el dataset.

# 46. Pipeline completo: versión inicial

In [ ]:
def pipeline_inicial(datos):
    resultado = datos.copy()

    resultado["importe"] = resultado.apply(
        lambda fila:
            fila["unidades"]
            * fila["precio"]
            * (
                1
                - fila["descuento"]
            ),
        axis=1
    )

    resultado = resultado.loc[
        resultado["importe"] > 100
    ].copy()

    return (
        resultado.groupby(
            "region",
            as_index=False
        )
        .agg(
            facturacion=("importe", "sum"),
            operaciones=("cliente_id", "size"),
            clientes=("cliente_id", "nunique")
        )
    )

# 47. Medir antes de optimizar

In [ ]:
datos_pipeline = df.head(
    100_000
)

bench_inicial = benchmark(
    lambda:
        pipeline_inicial(
            datos_pipeline
        ),
    repeticiones=3,
    calentamiento=0
)

print(
    bench_inicial["media_ms"],
    "ms"
)

# 48. Pipeline Pandas optimizado

In [ ]:
def pipeline_pandas_opt(datos):
    importe = (
        datos["unidades"]
        * datos["precio"]
        * (
            1
            - datos["descuento"]
        )
    )

    resultado = (
        datos.assign(
            importe=importe
        )
        .loc[
            lambda x:
                x["importe"] > 100
        ]
    )

    return (
        resultado.groupby(
            "region",
            as_index=False
        )
        .agg(
            facturacion=("importe", "sum"),
            operaciones=("cliente_id", "size"),
            clientes=("cliente_id", "nunique")
        )
    )

# 49. Pipeline Polars

In [ ]:
def pipeline_polars(datos_pl):
    return (
        datos_pl
        .with_columns(
            (
                pl.col("unidades")
                * pl.col("precio")
                * (
                    1
                    - pl.col("descuento")
                )
            )
            .alias("importe")
        )
        .filter(
            pl.col("importe") > 100
        )
        .group_by("region")
        .agg(
            pl.col("importe")
            .sum()
            .alias("facturacion"),

            pl.len()
            .alias("operaciones"),

            pl.col("cliente_id")
            .n_unique()
            .alias("clientes")
        )
    )

# 50. Comparar las tres versiones

In [ ]:
bench_pd_opt = benchmark(
    lambda:
        pipeline_pandas_opt(
            datos_pipeline
        ),
    repeticiones=7
)

if POLARS_DISPONIBLE:
    datos_pipeline_pl = pl.from_pandas(
        datos_pipeline
    )

    bench_polars = benchmark(
        lambda:
            pipeline_polars(
                datos_pipeline_pl
            ),
        repeticiones=7
    )
else:
    bench_polars = {
        "media_ms": np.nan
    }

comparacion_final = pd.DataFrame({
    "implementacion": [
        "Pandas inicial",
        "Pandas optimizado",
        "Polars"
    ],
    "media_ms": [
        bench_inicial["media_ms"],
        bench_pd_opt["media_ms"],
        bench_polars["media_ms"]
    ]
})

comparacion_final

# 51. Visualizar optimización completa

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(
    comparacion_final["implementacion"],
    comparacion_final["media_ms"]
)

ax.set(
    title="Evolución del pipeline",
    ylabel="Tiempo medio (ms)"
)

ax.tick_params(
    axis="x",
    rotation=15
)

plt.show()

# 52. Validar las tres implementaciones

In [ ]:
res_inicial = (
    pipeline_inicial(
        datos_pipeline
    )
    .sort_values("region")
    .reset_index(drop=True)
)

res_pd_opt = (
    pipeline_pandas_opt(
        datos_pipeline
    )
    .sort_values("region")
    .reset_index(drop=True)
)

print(
    "Inicial vs Pandas optimizado:",
    np.allclose(
        res_inicial["facturacion"],
        res_pd_opt["facturacion"]
    )
)

if POLARS_DISPONIBLE:
    res_pl = (
        pipeline_polars(
            datos_pipeline_pl
        )
        .sort("region")
        .to_pandas()
        .reset_index(drop=True)
    )

    print(
        "Pandas vs Polars:",
        np.allclose(
            res_pd_opt["facturacion"],
            res_pl["facturacion"]
        )
    )

### Regla fundamental

> Una optimización que cambia incorrectamente el resultado no es una optimización.

Rendimiento y corrección deben validarse conjuntamente.

# 53. Metodología profesional de optimización

Una secuencia recomendable es:

### 1. Definir el problema
¿Qué tarda demasiado? ¿Qué consume demasiada memoria?

### 2. Establecer una línea base
Medir antes de cambiar.

### 3. Perfilar
Encontrar dónde se consume realmente el tiempo.

### 4. Formular hipótesis
Por ejemplo: “`apply(axis=1)` domina el tiempo”.

### 5. Cambiar una cosa
Evitar múltiples optimizaciones simultáneas que impidan atribuir la mejora.

### 6. Volver a medir
Mismo dataset, mismas condiciones y suficientes repeticiones.

### 7. Validar
Comprobar equivalencia de resultados.

### 8. Analizar escalabilidad
Probar varios tamaños.

### 9. Documentar
Registrar contexto, hardware, versiones y metodología.

# 54. Checklist de benchmarking

- [ ] ¿Sé exactamente qué estoy midiendo?
- [ ] ¿Existe una línea base?
- [ ] ¿Utilizo un reloj de alta resolución?
- [ ] ¿Realizo calentamiento cuando procede?
- [ ] ¿Hago varias repeticiones?
- [ ] ¿Muestro dispersión?
- [ ] ¿Comparo resultados equivalentes?
- [ ] ¿Uso el mismo dataset?
- [ ] ¿Evito incluir conversiones solo en un motor?
- [ ] ¿Mido varios tamaños?
- [ ] ¿Analizo memoria además de tiempo?
- [ ] ¿He perfilado antes de optimizar?
- [ ] ¿He localizado el cuello de botella?
- [ ] ¿Valido el resultado después de optimizar?
- [ ] ¿La mejora compensa la complejidad añadida?

# 55. Ejercicios finales

## Ejercicio 1 — Big-O
Implementa ejemplos O(1), O(n) y O(n²) y representa su escalabilidad.

## Ejercicio 2 — Ruido
Mide una función 30 veces y representa la distribución de tiempos.

## Ejercicio 3 — `timeit`
Compara `perf_counter` y `timeit`.

## Ejercicio 4 — Vectorización
Transforma un loop numérico en una operación NumPy.

## Ejercicio 5 — Speedup
Calcula y explica el speedup.

## Ejercicio 6 — Pandas apply
Construye un `apply(axis=1)` deliberadamente lento y sustitúyelo.

## Ejercicio 7 — Escalabilidad
Compara ambas implementaciones en al menos cinco tamaños.

## Ejercicio 8 — cProfile
Perfila un pipeline y localiza las tres funciones con mayor `cumtime`.

## Ejercicio 9 — Memoria
Identifica la columna que más memoria consume y optimiza su dtype.

## Ejercicio 10 — Copias
Construye dos pipelines con diferente número de copias y compara el pico de memoria.

## Ejercicio 11 — Estructuras
Compara pertenencia en `list` y `set`.

## Ejercicio 12 — Pandas vs Polars
Compara una operación vectorizada y un groupby.

## Ejercicio 13 — Conversiones
Mide explícitamente Pandas → Polars y Polars → Pandas.

## Ejercicio 14 — Lazy
Compara una consulta Polars eager y lazy e inspecciona el plan.

## Ejercicio 15 — Escalabilidad de motores
Compara Pandas y Polars con cinco tamaños.

## Ejercicio 16 — Validación
Introduce deliberadamente una “optimización” que cambie el resultado y detecta el error mediante tests.

## Ejercicio 17 — Reto final
Construye un pipeline sobre al menos 500.000 filas que inicialmente sea ineficiente. Debes:

1. establecer baseline;
2. ejecutar al menos 10 repeticiones;
3. representar distribución;
4. ejecutar `cProfile`;
5. identificar cuello de botella;
6. medir memoria;
7. optimizar Pandas;
8. implementar Polars;
9. implementar Polars lazy;
10. medir conversiones;
11. probar cinco tamaños;
12. calcular speedups;
13. validar equivalencia;
14. representar tiempo;
15. representar memoria;
16. justificar qué versión llevarías a producción.

# 56. Preguntas de reflexión

1. ¿Qué describe realmente Big-O?
2. ¿Por qué Big-O no equivale a segundos?
3. ¿Por qué una medición única es insuficiente?
4. ¿Qué diferencia existe entre media y mediana de tiempos?
5. ¿Qué significa speedup?
6. ¿Por qué debemos incluir dispersión?
7. ¿Qué diferencia hay entre microbenchmark y profiling?
8. ¿Qué indica `cumtime`?
9. ¿Por qué `apply(axis=1)` suele ser una señal de revisión?
10. ¿Por qué vectorizar puede acelerar un cálculo?
11. ¿Qué coste tienen las copias?
12. ¿Por qué debemos estudiar varios tamaños?
13. ¿Cuándo una optimización no merece la pena?
14. ¿Por qué el coste de conversión importa al comparar motores?
15. ¿Qué aporta lazy execution?
16. ¿Qué condición debe cumplir siempre una optimización?
17. ¿Cuál debería ser el orden profesional: optimizar o perfilar?